# Monitoraggio della reputazione online di un'azienda

> **Azienda**: MachineInnovators Inc. — leader nello sviluppo di applicazioni di machine learning scalabili e pronte per la produzione

> **Problema**: monitorare manualmente il sentiment degli utenti sui social media e' inefficiente, soggetto a errori umani e troppo lento per intervenire in tempo su un calo di reputazione

> **Soluzione proposta**: automatizzare l'analisi del sentiment con un modello pre-addestrato e costruire attorno ad esso una pipeline MLOps reale di training, test, deploy e monitoraggio continuo

> **Modello richiesto**: [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)

> **Dataset pubblico**: [`tweet_eval`, task `sentiment`](https://huggingface.co/datasets/cardiffnlp/tweet_eval)

## Contesto

Il modo in cui un'azienda viene percepita sui social media influenza direttamente vendite, fiducia degli investitori e valore del brand. Il volume di menzioni (tweet, post, recensioni) che un'azienda di medie o grandi dimensioni riceve ogni giorno rende impossibile una lettura manuale sistematica: servono strumenti automatici capaci di leggere grandi quantita' di testo e restituire un giudizio sintetico e strutturato — questo e' il ruolo dell'NLP in questo progetto.

MachineInnovators Inc. vuole integrare metodologie **MLOps** per non fermarsi alla singola previsione, ma costruire un flusso completo che copra lo sviluppo, il testing, il deploy e il monitoraggio continuo del modello di analisi del sentiment. L'obiettivo di business e' abilitare l'azienda a migliorare e monitorare la propria reputazione sui social media in modo tempestivo — rilevando un peggioramento del sentiment mentre e' ancora gestibile, non dopo che e' gia' diventato un problema pubblico.

## Obiettivo del notebook

1. Caricare `tweet_eval/sentiment`, un dataset pubblico di tweet etichettati come `negative`/`neutral`/`positive`, e controllarne la qualita' (valori mancanti, duplicati, bilanciamento delle classi).
2. Caricare il modello pre-addestrato richiesto (`cardiffnlp/twitter-roberta-base-sentiment-latest`) ed eseguire l'inferenza su un campione del test set.
3. Valutare le prestazioni con metriche adatte a un problema multiclasse potenzialmente sbilanciato: accuracy, precision/recall/F1 macro e weighted, matrice di confusione, confidence.
4. Descrivere (non eseguire qui) la **pipeline CI/CD** automatizzata per training, test di integrazione e deploy, e il **sistema di monitoraggio continuo** — entrambi implementati come repository GitHub reale, non simulati nel notebook — con il link al repository pubblico richiesto dalla consegna.

## Metodologia

Il modello richiesto e' un Transformer (RoBERTa) gia' specializzato per la sentiment analysis su Twitter: viene usato **in inferenza diretta, senza fine-tuning**, e valutato sul benchmark pubblico `tweet_eval`. Il valore aggiunto del notebook non sta quindi nell'addestrare un classificatore da zero, ma nel valutare criticamente le prestazioni del modello gia' pronto.

I parametri di esecuzione (seed, batch size, numero di esempi valutati) sono centralizzati in un'unica `Config`: un solo punto da modificare invece di costanti sparse nel notebook.

Il training automatizzato, i test di integrazione, il deploy su HuggingFace e il monitoraggio continuo (Fase 2 e Fase 3 della consegna) **non vengono eseguiti in questo notebook**: sono implementati come pipeline reale nella repository GitHub (sezione 10), con codice che gira davvero tramite GitHub Actions — coerente con la consegna, che chiede una pipeline automatizzata e un sistema di monitoraggio, non una loro simulazione dentro un notebook.

## Struttura del notebook

0. Link al repository GitHub del progetto
1. Installazione librerie
2. Importazioni e configurazione centralizzata (`Config`)
3. Caricamento del dataset (`tweet_eval/sentiment`)
4. Controllo qualita' dei dati
5. Analisi esplorativa (distribuzione delle classi, lunghezza dei testi, esempi)
6. Caricamento del modello pre-addestrato
7. Funzioni modulari di inference e valutazione
8. Inference sul test set
9. Valutazione delle performance (metriche, matrice di confusione, confidence)
10. Pipeline CI/CD e monitoraggio continuo (descrizione della repository GitHub)
11. Conclusioni finali

Questo notebook e' pensato per essere eseguito su **Google Colab**.

## 0. Link repository GitHub

La consegna richiede che il notebook contenga il link al repository GitHub pubblico. Dopo aver creato il repository e caricato i file, sostituire il placeholder qui sotto.

In [ ]:
# Link al repository GitHub del progetto.
# Monorepo con tutti i progetti d'esame; questo link punta direttamente alla
# cartella con il codice di questo progetto (predictor, app, test, training,
# monitoraggio, pipeline CI/CD in .github/workflows/).
GITHUB_REPOSITORY_URL = "https://github.com/giuli-c/Folder-progetti-ProfessionAI/tree/main/Monitoraggio%20della%20reputazione%20online%20di%20un%E2%80%99azienda/sentiment_reputation_mlops"

print(f"Repository GitHub progetto: {GITHUB_REPOSITORY_URL}")

## 1. Installazione librerie

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib seaborn pandas numpy gradio huggingface_hub

## 2. Importazioni e configurazione

In [ ]:
# ============================================================
# IMPORTAZIONI E CONFIGURAZIONE
# ============================================================

import os
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

sns.set_theme(style="whitegrid")

In [ ]:
# ============================================================
# CELLA — Config: parametri centralizzati del progetto
# ============================================================
@dataclass
class Config:
    seed: int = 42
    model_name: str = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    dataset_name: str = "cardiffnlp/tweet_eval"
    dataset_task: str = "sentiment"
    max_eval_samples: int = 1000
    batch_size: int = 32
    label_map: Dict[int, str] = None

    def __post_init__(self):
        if self.label_map is None:
            # tweet_eval/sentiment usa questa codifica ufficiale:
            # 0 = negative, 1 = neutral, 2 = positive.
            self.label_map = {0: "negative", 1: "neutral", 2: "positive"}

cfg = Config()

In [ ]:
# ============================================================
# Impostazione del Seed
# ============================================================
def set_seed(seed: int = 42) -> None:
    """Rende piu' riproducibili campionamento e risultati."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
device = 0 if torch.cuda.is_available() else -1
print("Device usato:", "GPU" if device == 0 else "CPU")

## 3. Caricamento del dataset

Il dataset scelto e' `cardiffnlp/tweet_eval` (subtask `sentiment`), una raccolta pubblica di tweet etichettati come negativi, neutri o positivi. E' coerente con il modello CardiffNLP richiesto, perche' entrambi lavorano sul linguaggio tipico di Twitter/social media — non a caso condividono anche lo stesso namespace su HuggingFace Hub.

In [ ]:
# ============================================================
# CARICAMENTO DATASET PUBBLICO
# ============================================================

raw_dataset = load_dataset(cfg.dataset_name, cfg.dataset_task)

print(raw_dataset)
print("\nSplit disponibili:", list(raw_dataset.keys()))

In [ ]:
train_df = raw_dataset["train"].to_pandas()
val_df = raw_dataset["validation"].to_pandas()
test_df = raw_dataset["test"].to_pandas()

for df in [train_df, val_df, test_df]:
    df["sentiment"] = df["label"].map(cfg.label_map)
    df["text_length"] = df["text"].str.len()
    df["word_count"] = df["text"].str.split().str.len()

print("Dimensioni train/validation/test:")
print(train_df.shape, val_df.shape, test_df.shape)

train_df.head()

## 4. Controllo qualita' dei dati

Prima di usare un modello e' utile controllare valori mancanti, duplicati e distribuzione delle etichette. 

In [ ]:
# ============================================================
# DATA QUALITY CHECK
# ============================================================

def data_quality_report(df: pd.DataFrame, name: str) -> None:
    print(f"--- {name.upper()} ---")
    print(f"Righe: {len(df):,}")
    print("Valori mancanti:")
    print(df.isnull().sum())
    print(f"Duplicati sul testo: {df['text'].duplicated().sum():,}")
    print("Distribuzione label:")
    print(df["sentiment"].value_counts().to_string())
    print()

data_quality_report(train_df, "train")
data_quality_report(val_df, "validation")
data_quality_report(test_df, "test")

## 5. Analisi esplorativa

Questa sezione serve a capire la forma dei dati prima della valutazione del modello: quante classi ci sono, quanto sono bilanciate e quanto sono lunghi i testi. Sono informazioni semplici, ma aiutano molto a interpretare i risultati successivi.

In [ ]:
# ============================================================
# GRAFICO 1: DISTRIBUZIONE DELLE CLASSI
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, df) in zip(axes, [("Train", train_df), ("Validation", val_df), ("Test", test_df)]):
    order = ["negative", "neutral", "positive"]
    counts = df["sentiment"].value_counts().reindex(order)
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette="Set2")
    ax.set_title(f"Distribuzione sentiment - {name}")
    ax.set_xlabel("Sentiment")
    ax.set_ylabel("Numero testi")
    for i, val in enumerate(counts.values):
        ax.text(i, val + max(counts.values) * 0.02, f"{val:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## Analisi della qualità e della distribuzione del dataset

### Controllo degli split

Il dataset è suddiviso in:

- **Train:** 45.615 esempi
- **Validation:** 2.000 esempi
- **Test:** 12.284 esempi

Il controllo preliminare non evidenzia **valori mancanti** nelle variabili analizzate (`text`, `label`, `sentiment`, `text_length`, `word_count`) in nessuno dei tre split.

Sono stati rilevati **29 testi duplicati nel Train**, mentre Validation e Test non presentano duplicati. Considerata la dimensione del training set, si tratta comunque di una quota molto ridotta.

### Distribuzione delle classi

| Split | Negative | Neutral | Positive |
|---|---:|---:|---:|
| **Train** | 7.093 (15,5%) | 20.673 (45,3%) | 17.849 (39,1%) |
| **Validation** | 312 (15,6%) | 869 (43,5%) | 819 (41,0%) |
| **Test** | 3.972 (32,3%) | 5.937 (48,3%) | 2.375 (19,3%) |

Guardando le percentuali, Train e Validation si assomigliano molto: in entrambi la classe più comune è `neutral`, poi `positive`, e `negative` è quella con meno esempi.

Il **Test set presenta invece una composizione sensibilmente diversa**. 

La classe `negative` passa da circa **15–16% a 32,3%**, quindi quasi raddoppia, mentre `positive` scende da circa **39–41% a 19,3%**, circa la metà. `Neutral` rimane invece la classe più rappresentata in tutti gli split.

→ È quindi presente un **distribution shift nelle proporzioni delle classi tra Train/Validation e Test**.

Questo aspetto dovrà essere considerato nell'interpretazione delle metriche finali, perché il modello verrà valutato su una distribuzione delle label diversa da quella osservata durante training e validation.

In [ ]:
# ============================================================
# GRAFICO 2: LUNGHEZZA DEI TESTI PER SENTIMENT
# ============================================================

plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="sentiment", y="word_count", order=["negative", "neutral", "positive"], palette="Set2")
plt.title("Distribuzione della lunghezza dei testi per sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Numero parole")
plt.tight_layout()
plt.show()

### Osservazioni sulla lunghezza dei testi

Dal grafico si può osservare che la **lunghezza dei testi è abbastanza simile nelle tre classi di sentiment**.

- I testi `negative` hanno una lunghezza mediana leggermente maggiore, circa **21 parole**.
- I testi `neutral` e `positive` hanno invece una mediana di circa **19 parole**.
- La maggior parte dei testi, per tutte le classi, si concentra indicativamente tra **16 e 24 parole**.
- Sono presenti alcuni **valori anomali (outlier)**, soprattutto testi molto brevi con meno di 5-6 parole e alcuni testi più lunghi, intorno alle 34-35 parole.

Nel complesso, non si notano grandi differenze nella lunghezza dei testi tra `negative`, `neutral` e `positive`.

→ La **lunghezza del testo non sembra quindi essere una caratteristica che distingue in modo evidente le tre classi**: il sentiment dovrà essere riconosciuto principalmente a partire dal contenuto delle frasi e non semplicemente dal numero di parole.

In [ ]:
# ============================================================
# GRAFICO 3: ESEMPI DI TESTI PER CLASSE
# ============================================================

for sentiment in ["negative", "neutral", "positive"]:
    print(f"\n--- Esempi classe {sentiment.upper()} ---")
    sample_texts = train_df[train_df["sentiment"] == sentiment].sample(3, random_state=cfg.seed)["text"].tolist()
    for idx, text in enumerate(sample_texts, start=1):
        print(f"{idx}. {text}")

### Osservazioni sugli esempi testuali

Osservando alcuni esempi casuali per ciascuna classe si nota che i testi hanno le caratteristiche tipiche di messaggi provenienti da **Twitter/X**: sono brevi, informali e possono contenere **hashtag, menzioni (`@user`), abbreviazioni, nomi propri e punteggiatura usata per enfatizzare il messaggio**.

Negli esempi `negative` sono presenti espressioni che comunicano chiaramente insoddisfazione o frustrazione, mentre nei `positive` compaiono messaggi con un tono più favorevole o entusiasta. I testi `neutral` risultano invece più descrittivi o informativi e non esprimono un'opinione particolarmente positiva o negativa.

Si nota inoltre che il sentiment non dipende necessariamente da singole parole, ma dal **significato complessivo della frase e dal contesto**.

→ Questi esempi mostrano quindi che la classificazione del sentiment richiede di comprendere il contenuto del testo, tenendo conto anche del linguaggio informale tipico dei social network.

## 6. Caricamento del modello pre-addestrato

Il modello richiesto e' una versione RoBERTa addestrata per sentiment analysis su Twitter. Usiamo una pipeline HuggingFace per tenere il codice leggibile e adatto a Colab.

In [ ]:
# ============================================================
# MODELLO HUGGINGFACE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name)

# Creazione una pipeline Hugging Face per la classificazione del sentiment.
# La pipeline si occupa automaticamente di:
# - tokenizzare il testo con il tokenizer scelto;
# - passare gli input al modello;
# - eseguire la previsione;
# - restituire la classe prevista con il relativo score.
#
# `task` indica a Hugging Face quale operazione vogliamo eseguire.
# Alcuni esempi:
# - task="text-classification"     -> classificazione generica di un testo
#   es. "This email is spam" -> spam
# - task="text-generation"         -> generazione di testo
#   es. "Once upon a time..." -> continua la frase
# - task="summarization"           -> riassunto di un testo
#   es. testo lungo -> breve riassunto
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=128,
)

print("Modello caricato:", cfg.model_name)
print("Etichette modello:", model.config.id2label)

## 7. Funzioni modulari di inference e valutazione

In [ ]:
# ============================================================
# FUNZIONI DI UTILITA'
# ============================================================

def normalize_model_label(label: str) -> str:
    """Converte eventuali label HuggingFace tipo LABEL_0 nei nomi sentiment."""
    label = label.lower()
    if label.startswith("label_"):
        label_id = int(label.replace("label_", ""))
        return cfg.label_map[label_id]
    return label

def predict_sentiment(texts: List[str], batch_size: int = 32, model_pipeline=None) -> Tuple[List[str], List[float]]:
    """
    Predice sentiment e confidence score per una lista di testi.
    Usa `sentiment_pipeline` (il modello originale) di default; passare
    `model_pipeline` per riusare la stessa logica di inferenza con un altro
    modello.
    """
    pipe = model_pipeline if model_pipeline is not None else sentiment_pipeline
    predictions = []
    scores = []

    for start in range(0, len(texts), batch_size):
        # prendo una porzione della lista texts
        # ["frase 0", "frase 1"]
        batch = texts[start:start + batch_size]
        outputs = pipe(batch, batch_size=batch_size)
        for out in outputs:
            if start in range(0, 10):
              print(out)
            predictions.append(normalize_model_label(out["label"]))
            scores.append(float(out["score"]))

    return predictions, scores

def evaluate_sentiment_model(df: pd.DataFrame, split_name: str) -> Dict[str, float]:
    """
    Valuta il modello su uno split e restituisce metriche principali.
    """
    y_true = df["sentiment"].tolist()
    y_pred = df["predicted_sentiment"].tolist()

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

def sample_for_test(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """
    Campiona lo split per rendere l'esecuzione sostenibile su Colab.
    """
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

## 8. Inference sul test set

Per mantenere il notebook veloce in Colab, la valutazione usa un campione del test set. In un ambiente di produzione useremmo l'intero test set o una suite di benchmark versionata.

In [ ]:
# ============================================================
# PREDIZIONE SUL TEST SET
# ============================================================

sample_test_df = sample_for_test(test_df, cfg.max_eval_samples, cfg.seed)

start_time = time.time()
preds, scores = predict_sentiment(sample_test_df["text"].tolist(), batch_size=cfg.batch_size)
elapsed = time.time() - start_time

sample_test_df["predicted_sentiment"] = preds
sample_test_df["confidence"] = scores

print(f"Esempi valutati: {len(sample_test_df):,}")
print(f"Tempo inference: {elapsed:.2f} secondi")
print(f"Tempo medio per testo: {elapsed / len(sample_test_df):.4f} secondi")

sample_test_df[["text", "sentiment", "predicted_sentiment", "confidence"]].head()

## 9. Valutazione delle performance

Usiamo piu' metriche per evitare una lettura troppo superficiale. L'accuracy e' intuitiva, ma con tre classi e possibile sbilanciamento e' importante guardare anche macro precision, macro recall e macro F1.

In [ ]:
# ============================================================
# METRICHE DI VALUTAZIONE
# ============================================================

metrics = evaluate_sentiment_model(sample_test_df, "test_sample")
metrics_df = pd.DataFrame([metrics]).set_index("split")
display(metrics_df.style.format("{:.4f}"))

print("Classification report:\n")
print(classification_report(
    sample_test_df["sentiment"],
    sample_test_df["predicted_sentiment"],
    labels=["negative", "neutral", "positive"],
    zero_division=0,
))

In [ ]:
# ============================================================
# GRAFICO 4: METRICHE PRINCIPALI
# ============================================================

metric_order = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]
plot_metrics = metrics_df.loc["test_sample", metric_order]

plt.figure(figsize=(10, 4))
sns.barplot(x=plot_metrics.index, y=plot_metrics.values, palette="viridis")
plt.ylim(0, 1)
plt.title("Metriche di performance sul campione di test")
plt.ylabel("Score")
plt.xlabel("Metrica")
plt.xticks(rotation=20, ha="right")
for i, val in enumerate(plot_metrics.values):
    plt.text(i, val + 0.02, f"{val:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

### Osservazione sulle metriche

Il modello raggiunge un'**accuracy del 70%** sul campione di test, con valori simili anche per le altre metriche principali:

- **Precision macro:** 0.699
- **Recall macro:** 0.710
- **F1-score macro:** 0.702
- **F1-score weighted:** 0.698

Le prestazioni risultano quindi **abbastanza bilanciate tra le classi**.

Analizzando i risultati per singola classe:

- **Negative** → è la classe riconosciuta meglio, con **F1-score = 0.73** e **recall = 0.79**.
- **Positive** → mostra prestazioni equilibrate, con **F1-score = 0.70**.
- **Neutral** → risulta la classe più difficile da identificare, con **recall = 0.63** e **F1-score = 0.67**.

Nel complesso, il modello mostra **prestazioni discrete e abbastanza uniformi**, ma presenta maggiore difficoltà nel riconoscimento dei testi **neutrali**.

In [ ]:
# ============================================================
# GRAFICO 5: MATRICE DI CONFUSIONE
# ============================================================

labels = ["negative", "neutral", "positive"]
cm = confusion_matrix(sample_test_df["sentiment"], sample_test_df["predicted_sentiment"], labels=labels)
cm_df = pd.DataFrame(cm, index=[f"Reale {l}" for l in labels], columns=[f"Pred {l}" for l in labels])

plt.figure(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matrice di confusione - sentiment")
plt.tight_layout()
plt.show()

### Osservazione sulla matrice di confusione

La matrice di confusione permette di osservare **quali classi vengono riconosciute correttamente e quali vengono confuse tra loro**.

- **Negative:** 259 esempi su 328 vengono classificati correttamente. L'errore principale è la classificazione come **neutral** (63 casi).
- **Neutral:** 293 esempi su 462 sono corretti. È la classe che genera più errori: 110 esempi vengono classificati come **negative** e 59 come **positive**.
- **Positive:** 148 esempi su 210 vengono classificati correttamente. Gli errori sono soprattutto verso la classe **neutral** (53 casi), mentre solo 9 vengono classificati come **negative**.

Nel complesso, il modello distingue abbastanza bene i sentimenti **negative** e **positive**, mentre mostra maggiore difficoltà con la classe **neutral**, che tende a sovrapporsi alle altre due.

È inoltre interessante notare che la confusione diretta tra **positive** e **negative** è molto bassa (6 e 9 casi): gli errori avvengono soprattutto passando attraverso la classe **neutral**.

In [ ]:
# ============================================================
# GRAFICO 6: CONFIDENCE DEL MODELLO
# ============================================================

plt.figure(figsize=(10, 5))
sns.histplot(data=sample_test_df, x="confidence", hue="predicted_sentiment", bins=25, kde=True, palette="Set2")
plt.title("Distribuzione della confidence per sentiment predetto")
plt.xlabel("Confidence del modello")
plt.ylabel("Numero testi")
plt.tight_layout()
plt.show()

### Osservazione sulla confidence del modello

Il grafico mostra **quanto il modello è sicuro delle proprie predizioni**, distinguendo i risultati in base al sentiment predetto.

Per leggere il grafico:
- sull'**asse X** è riportata la **confidence**, cioè il livello di sicurezza associato alla previsione: valori più vicini a **1** indicano una maggiore sicurezza;
- sull'**asse Y** è riportato il **numero di testi** che presentano un determinato intervallo di confidence;
- i diversi **colori** rappresentano le tre classi predette: `negative`, `neutral` e `positive`;
- le **linee** aiutano a visualizzare l'andamento generale della distribuzione della confidence per ciascuna classe.

Dal grafico emerge che le predizioni **positive** sono spesso associate a confidence molto elevate, concentrate soprattutto verso **0.90–1.00**. Anche per la classe **negative** il modello mostra generalmente una buona sicurezza, con numerose predizioni ad alta confidence.

La classe **neutral** presenta invece una distribuzione più ampia, con molti valori anche tra circa **0.50 e 0.80**. Questo indica che il modello tende ad essere **meno sicuro quando assegna un testo alla classe neutral**.

Il risultato è coerente con le analisi precedenti: la classe `neutral` è quella che presenta il **recall e l'F1-score più bassi** e, nella matrice di confusione, è anche quella maggiormente confusa con le altre classi.

> **Nota:** una confidence elevata indica che il modello è molto sicuro della propria previsione, ma **non garantisce che la previsione sia corretta**. 

## 10. Pipeline CI/CD e monitoraggio continuo

Le Fasi 2 e 3 della consegna (pipeline automatizzata per training/test/deploy, sistema di monitoraggio continuo) non sono simulazioni dentro questo notebook: sono implementate come repository GitHub reale, con codice che gira davvero tramite GitHub Actions. Qui vengono solo descritte; il codice vive in [`sentiment_reputation_mlops/`](sentiment_reputation_mlops/) e nei workflow alla radice del repository (vedi struttura sotto).

In [ ]:
# ============================================================
# STRUTTURA DEL REPOSITORY (riferimento — i file vivono nel repo, non qui)
# ============================================================

REPO_STRUCTURE = """
<radice del repository GitHub>
├── .github/workflows/
│   ├── ci.yml                # job "test" (pytest) + job "deploy" (HuggingFace Space, dopo i test)
│   ├── train.yml             # job "train", trigger manuale (workflow_dispatch)
│   └── monitor.yml           # job "monitor", schedulato (cron) + trigger manuale
└── sentiment_reputation_mlops/
    ├── requirements.txt      # dipendenze del repository (transformers, gradio, requests, ecc.)
    ├── config.py             # costanti centralizzate: modello, dataset, soglie, repo HuggingFace
    ├── predictor.py          # SentimentPredictor: carica il modello una volta, espone predict()
    ├── app.py                # demo Gradio, usa SentimentPredictor da predictor.py
    ├── train.py              # retraining su dati mai visti dal modello base + gate di promozione
    ├── monitor.py            # monitoraggio del sentiment su post reali (Mastodon), con baseline storica
    ├── deploy_to_hf.py       # pubblica questa cartella come HuggingFace Space
    ├── README.md             # frontmatter richiesto da HuggingFace Space
    ├── conftest.py           # vuoto: serve solo perche' pytest trovi predictor.py da tests/
    ├── monitoring/history.json  # baseline storica, aggiornata automaticamente dal job "monitor"
    └── tests/
        ├── test_smoke.py     # test_model_loads, test_known_examples, casi limite, schema di output
        └── test_app.py       # verifica che app.py (Gradio) funzioni, non solo predictor.py
"""

print(REPO_STRUCTURE)


**Osservazione sulla pipeline CI/CD e sul monitoraggio**

**`config.py`** centralizza le costanti usate dagli altri script (nome del modello, dataset di retraining, repository HuggingFace di destinazione, soglie) — stesso principio della `Config` del notebook (sezione 2), ma per il codice che vive nella repository.

**Job `test`** (`ci.yml`): installa le dipendenze ed esegue `pytest` ad ogni push o pull request su `main` che tocchi `sentiment_reputation_mlops/`. `app.py` e `tests/test_smoke.py` importano entrambi `SentimentPredictor` da `predictor.py`, che accentra il caricamento del modello in un solo posto. Oltre allo smoke test, `test_smoke.py` verifica lo schema di output (etichetta valida, confidence in [0, 1]) su casi limite (testo vuoto, molto lungo, lingua diversa dall'inglese, emoji); `test_app.py` verifica che anche `app.py` — non solo `predictor.py` — funzioni davvero.

**Job `deploy`** (`ci.yml`, `needs: test`, solo su push a `main`): pubblica `app.py` come HuggingFace Space tramite `huggingface_hub`, creandolo al primo deploy se non esiste ancora.

**Job `train`** (`train.yml`, trigger manuale `workflow_dispatch`): esegue `train.py`, che riallena il modello su `mteb/tweet_sentiment_extraction` — un dataset **diverso** da quello di valutazione (`tweet_eval`), perche' il modello base e' gia' stato fine-tuned proprio su TweetEval (lo dice la sua model card su HuggingFace): riallenarlo sugli stessi dati non introdurrebbe nessuna informazione nuova. Il confronto prima/dopo viene fatto sia sul dataset nuovo sia su un campione di `tweet_eval` (controllo di regressione, per verificare che il modello non abbia "dimenticato" quello che sapeva gia' fare bene). Il modello riaddestrato viene pubblicato su un repository HuggingFace dedicato **solo se** il calo di F1 macro sul benchmark originale resta entro una soglia di tolleranza: altrimenti il job fallisce esplicitamente e non pubblica nulla. La promozione a "modello in produzione" resta comunque una decisione manuale.

**Job `monitor`** (`monitor.yml`, schedulato una volta al giorno + trigger manuale): esegue `monitor.py`, che scarica testi pubblici reali da un'istanza Mastodon (nessuna etichetta necessaria: il monitoraggio del drift di sentiment si basa solo sulle predizioni del modello, non sull'accuratezza), li classifica con il modello attuale, calcola la quota di sentiment negativo del batch e la confronta con una baseline storica (media + 1 deviazione standard delle esecuzioni precedenti). Il risultato viene salvato in `monitoring/history.json`, che il workflow ricommitta nel repository ad ogni esecuzione: e' cosi' che la baseline cresce davvero nel tempo, invece di essere ricalcolata da zero ogni volta.

In un progetto reale aggiungerei anche controlli sul formato dei dati e linting nella pipeline di test; il punto importante resta che il modello non dovrebbe arrivare in produzione solo perche' il notebook funziona: serve una catena automatica che controlli codice, dipendenze e comportamento minimo del sistema nel tempo.

## 11. Conclusioni finali

Il progetto usa un modello di sentiment analysis gia' pronto (CardiffNLP, addestrato su testi social) per classificare testi in `negative`/`neutral`/`positive`, e ne valuta criticamente le prestazioni su un benchmark pubblico (`tweet_eval`) prima di considerarlo adatto all'uso.

Le Fasi 2 e 3 della consegna — pipeline CI/CD automatizzata e sistema di monitoraggio continuo — non sono descritte solo a parole: sono implementate come repository GitHub reale (sezione 10), con codice che gira davvero tramite GitHub Actions:

- **training** automatizzato su un dataset diverso da quello di valutazione (per introdurre davvero informazione nuova al modello), con pubblicazione del modello condizionata a un controllo di regressione;
- **test di integrazione** su schema di output, casi limite, e sull'applicazione stessa (`app.py`), non solo sul modello;
- **deploy** automatico su HuggingFace Space dopo il superamento dei test;
- **monitoraggio continuo** del sentiment su dati social reali (Mastodon), con una baseline storica che si aggiorna da sola nel repository nel tempo.